# SVARA — Google Colab GPU Training Runner
**Smart Voice Assistant for Residential Automation**  
Informatics Engineering, Universitas Mercu Buana  

This notebook trains and benchmarks the main model **M0 (`facebook/wav2vec2-base`)** on Google Colab GPU.

### Prerequisites:
- Colab Runtime set to **GPU (T4 or A100)** (`Runtime > Change runtime type > T4 GPU`).
- Fluent Speech Commands dataset placed in Google Drive at `MyDrive/SVARA/data/raw/fluent_speech_commands_dataset` (or downloaded directly).

## 1. Environment & GPU Check

In [1]:
!nvidia-smi

Fri Oct  9 18:44:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 573.22                 Driver Version: 573.22         CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   50C    P8              3W /   75W |     146MiB /   8188MiB |      5%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Mount Google Drive & Clone/Update SVARA Repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Set HuggingFace cache directory on Drive to avoid re-downloading checkpoints
os.environ['HF_HOME'] = '/content/drive/MyDrive/SVARA/cache/huggingface'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

%cd /content
if not os.path.exists('SVARA'):
    !git clone https://github.com/YogUNI/SVARA.git
    %cd SVARA
else:
    %cd SVARA
    !git pull origin main

## 3. Install Requirements

In [ ]:
!pip install -r requirements.txt
!pip install -e .

## 4. Link or Copy Dataset & Splits
Symlink the raw FSC dataset and pre-computed splits from Drive if already extracted.

In [ ]:
import os
# Check if dataset exists in Drive
drive_dataset = '/content/drive/MyDrive/SVARA/data/raw/fluent_speech_commands_dataset'
local_data_dir = 'data/raw/fluent_speech_commands_dataset'

if os.path.exists(drive_dataset) and not os.path.exists(local_data_dir):
    os.makedirs('data/raw', exist_ok=True)
    !ln -s "$drive_dataset" "$local_data_dir"
    print("Linked dataset from Google Drive successfully!")
else:
    print("Local dataset or drive status:", os.path.exists(local_data_dir))

# Run data check audit
!python scripts/00_check_data.py --root data/raw/fluent_speech_commands_dataset
# Generate splits if not already present
!python scripts/01_make_splits.py --config configs/splits.yaml

## 5. Sanity Checks Verification (docs/11 V-10..V-20)
Ensure zero silent bugs before commencing long GPU training.

In [2]:
!python scripts/sanity_checks.py --config configs/model_w2v2_base.yaml

python: can't open file 'c:\\Users\\Axioo Pongo\\SVARA\\notebooks\\scripts\\sanity_checks.py': [Errno 2] No such file or directory


## 6. Train M0 (`facebook/wav2vec2-base`) — Split A & Split B1 (Tasks P3-07, P3-08)
The training loop is resumable (`--resume`) and automatically saves checkpoints to `reports/runs/`.

In [2]:
%cd ..
!python scripts/train.py --config configs/model_w2v2_base.yaml --split A --seed 42

c:\Users\Axioo Pongo\SVARA
[train.py] Using device: cuda | Seed: 42 | Split: A
Epoch 01/15 | Train Loss: 2.9195 | Train Acc: 0.320 | Val Loss: 0.4653 | Val Acc: 0.892 | Time: 793.33s
Epoch 02/15 | Train Loss: 0.2103 | Train Acc: 0.954 | Val Loss: 0.3549 | Val Acc: 0.941 | Time: 663.96s
Epoch 03/15 | Train Loss: 0.1298 | Train Acc: 0.973 | Val Loss: 0.2074 | Val Acc: 0.967 | Time: 657.5s
Epoch 04/15 | Train Loss: 0.1029 | Train Acc: 0.980 | Val Loss: 0.2623 | Val Acc: 0.963 | Time: 1038.69s
Epoch 05/15 | Train Loss: 0.0790 | Train Acc: 0.984 | Val Loss: 0.2239 | Val Acc: 0.963 | Time: 901.84s
Epoch 06/15 | Train Loss: 0.0746 | Train Acc: 0.986 | Val Loss: 0.1314 | Val Acc: 0.979 | Time: 824.45s
Epoch 07/15 | Train Loss: 0.0597 | Train Acc: 0.987 | Val Loss: 0.2301 | Val Acc: 0.971 | Time: 930.49s
Epoch 08/15 | Train Loss: 0.0526 | Train Acc: 0.988 | Val Loss: 0.2222 | Val Acc: 0.973 | Time: 1176.84s
Epoch 09/15 | Train Loss: 0.0473 | Train Acc: 0.990 | Val Loss: 0.1837 | Val Acc: 0.974 


Loading weights: 100%|██████████| 211/211 [00:00<00:00, 42030.69it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.

Training: 100%|██████████| 1446/1446 [12:01<00:00,  1.69it/s, loss=2.9295, acc=0.317]
                                                                                     

Training: 100%|██████████| 1446/1446 [10:07<00:00,  2.47it/s, loss=0.2106, acc=0.954]
                                              

In [1]:
import os
os.chdir(r"C:\Users\Axioo Pongo\SVARA")

!python scripts/train.py --config configs/model_w2v2_base.yaml --split B1 --seed 42 --resume --output-dir reports/runs/20261010-0128_wav2vec2_B1_s42

[train.py] Using device: cuda | Seed: 42 | Split: B1
[train.py] Resuming from checkpoint: reports/runs/20261010-0128_wav2vec2_B1_s42\last.ckpt
Epoch 02/15 | Train Loss: 0.2072 | Train Acc: 0.957 | Val Loss: 0.7308 | Val Acc: 0.872 | Time: 736.57s
Epoch 03/15 | Train Loss: 0.1238 | Train Acc: 0.976 | Val Loss: 0.3733 | Val Acc: 0.911 | Time: 553.54s
Epoch 04/15 | Train Loss: 0.0973 | Train Acc: 0.980 | Val Loss: 1.0260 | Val Acc: 0.839 | Time: 439.33s
Epoch 05/15 | Train Loss: 0.0794 | Train Acc: 0.984 | Val Loss: 0.5763 | Val Acc: 0.866 | Time: 504.64s
Epoch 06/15 | Train Loss: 0.0689 | Train Acc: 0.986 | Val Loss: 0.9414 | Val Acc: 0.853 | Time: 434.52s
[train.py] Early stopping triggered after 6 epochs (patience=3).
[train.py] Training completed. Best Val Acc: 0.9112 | Run directory: reports/runs/20261010-0128_wav2vec2_B1_s42



Loading weights: 100%|██████████| 211/211 [00:00<00:00, 9879.97it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.

Training: 100%|██████████| 1348/1348 [10:44<00:00,  2.79it/s, loss=0.2073, acc=0.957]
                                                                                     

Training: 100%|██████████| 1348/1348 [07:33<00:00,  2.54it/s, loss=0.1245, acc=0.976]
                                               

## 7. Evaluate Runs (Task P3-05)
Compute exact-match accuracy, per-slot accuracy, bootstrap 95% CI, and confusion matrices.

In [2]:
import glob
runs = sorted(glob.glob('reports/runs/*_wav2vec2_*'))
for r in runs:
    print(f"Evaluating {r}...")
    split = 'A' if '_A_' in r else 'B1'
    !python scripts/evaluate.py --run "$r" --split "$split" --set test

Evaluating reports/runs\20261009-1851_wav2vec2_A_s42...
[evaluate.py] Evaluating run reports/runs\20261009-1851_wav2vec2_A_s42 on Split A (test)...
[evaluate.py] Loaded weights from reports/runs\20261009-1851_wav2vec2_A_s42\best.ckpt
[evaluate.py] Done! Exact-match: 0.9533 (95% CI: [0.9465, 0.9597])
[evaluate.py] Saved to: reports/runs\20261009-1851_wav2vec2_A_s42\metrics_A_test.json
Evaluating reports/runs\20261009-1953_wav2vec2_A_s42...



Loading weights: 100%|██████████| 211/211 [00:00<00:00, 40464.46it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.bias             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[evaluate.py] Evaluating run reports/runs\20261009-1953_wav2vec2_A_s42 on Split A (test)...
[evaluate.py] Loaded weights from reports/runs\20261009-1953_wav2vec2_A_s42\best.ckpt
[evaluate.py] Done! Exact-match: 0.9960 (95% CI: [0.9939, 0.9979])
[evaluate.py] Saved to: reports/runs\20261009-1953_wav2vec2_A_s42\metrics_A_test.json
Evaluating reports/runs\20261010-0128_wav2vec2_B1_s42...



Loading weights: 100%|██████████| 211/211 [00:00<00:00, 32146.68it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[evaluate.py] Evaluating run reports/runs\20261010-0128_wav2vec2_B1_s42 on Split B1 (test)...
[evaluate.py] Loaded weights from reports/runs\20261010-0128_wav2vec2_B1_s42\best.ckpt
[evaluate.py] Done! Exact-match: 0.9587 (95% CI: [0.9533, 0.9638])
[evaluate.py] Saved to: reports/runs\20261010-0128_wav2vec2_B1_s42\metrics_B1_test.json
Evaluating reports/runs\20261010-0836_wav2vec2_B1_s42...



Loading weights: 100%|██████████| 211/211 [00:00<00:00, 28529.92it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.bias             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[evaluate.py] Evaluating run reports/runs\20261010-0836_wav2vec2_B1_s42 on Split B1 (test)...
[evaluate.py] WARNING: No checkpoint found in reports/runs\20261010-0836_wav2vec2_B1_s42, using initialized weights.
[evaluate.py] Done! Exact-match: 0.0083 (95% CI: [0.0062, 0.0108])
[evaluate.py] Saved to: reports/runs\20261010-0836_wav2vec2_B1_s42\metrics_B1_test.json



Loading weights: 100%|██████████| 211/211 [00:00<00:00, 37010.63it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.bias               | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 8. Backup Runs & Checkpoints to Google Drive
Ensures training results and checkpoints are safe and persistent.

In [ ]:
!mkdir -p /content/drive/MyDrive/SVARA/reports/
!cp -r reports/runs /content/drive/MyDrive/SVARA/reports/
print("Backed up all run logs and checkpoints to Google Drive successfully!")